# Full HotpotQA run in Google Colab

This notebook converts the **complete** HotpotQA train and validation parquet
shards, trains the LiteRag EXPAND/STOP policy, and evaluates retrieval on every
validation question (the project's `dev` split).

## Before running

1. In Colab, choose a **High-RAM runtime** (at least 40 GiB total; 50 GiB is
   recommended). The current fallback creates dense 2,048-dimensional float32
   embeddings: about 14.4 GiB for ~1.89M sentence chunks plus at least ~3.9 GiB
   for title entities (aliases can increase this), before Python objects and working memory. A GPU is not required by
   this NumPy implementation.
2. Put the three source files in Google Drive under
   `MyDrive/HotpotQA/data/`:
   `train-00000-of-00002.parquet`, `train-00001-of-00002.parquet`, and
   `validation-00000-of-00001.parquet`.
3. Push the project files to GitHub first, or edit the clone cell to use your
   repository/branch. The data files need not be committed.
4. Run all cells in order. The full conversion and evaluation can take a long
   time. Outputs are saved under `MyDrive/LiteRag_Full_Run/`.

The default policy training uses 1,000 train questions for 2 epochs to keep the
run tractable, but retrieval evaluation uses the **entire** validation split.
Set `TRAIN_LIMIT = None` to train on all train questions; that can take many
hours because the current implementation scores the full chunk corpus per
question.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 2. Clone the project and install required packages


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CharanGowda-006/LiteRag.git"
REPO_DIR = Path("/content/LiteRag")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy", "pandas", "pyarrow", "psutil",
], check=True)

import os
os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)


## 3. Set paths and check runtime memory


In [ ]:
import glob
import psutil

DATA_DIR = Path("/content/drive/MyDrive/HotpotQA/data")
RUN_DIR = Path("/content/drive/MyDrive/LiteRag_Full_Run")
KG_DIR = RUN_DIR / "data"
POLICY_PATH = RUN_DIR / "policy.npz"
PARAMS_PATH = RUN_DIR / "rl_params.json"
METRICS_DIR = RUN_DIR / "metrics"
RUN_DIR.mkdir(parents=True, exist_ok=True)

train_files = sorted(DATA_DIR.glob("train-*.parquet"))
validation_files = sorted(DATA_DIR.glob("validation-*.parquet"))
assert len(train_files) == 2, f"Expected 2 train shards in {DATA_DIR}, found {len(train_files)}"
assert len(validation_files) == 1, f"Expected 1 validation shard in {DATA_DIR}, found {len(validation_files)}"

memory = psutil.virtual_memory()
print(f"RAM total: {memory.total / 1024**3:.1f} GiB; available: {memory.available / 1024**3:.1f} GiB")
if memory.total < 40 * 1024**3:
    raise RuntimeError("Select a Colab High-RAM runtime (at least 40 GiB; 50 GiB recommended).")
if memory.available < 30 * 1024**3:
    raise RuntimeError("Not enough free RAM for the full dense embedding matrices; restart with High-RAM.")

print("Train shards:", [p.name for p in train_files])
print("Validation shards:", [p.name for p in validation_files])
print("Output directory:", RUN_DIR)


## 4. Convert all train and validation rows


In [ ]:
subprocess.run([
    sys.executable,
    str(REPO_DIR / "kg_interface" / "build_hotpot_kg.py"),
    "--data", str(DATA_DIR),
    "--out", str(KG_DIR),
    "--sample-train", "0",
    "--sample-validation", "0",
], check=True)


## 5. Validate the generated graph interface

The converter maps the full training split to `train` and full validation split
to `dev`. HotpotQA validation is the held-out evaluation split here; the
source files do not provide a separate labeled test split.


In [ ]:
subprocess.run([
    sys.executable,
    str(REPO_DIR / "kg_interface" / "validate_kg.py"),
    str(KG_DIR),
], check=True)


## 6. Load the full graph and train the policy


In [ ]:
import json
import sys

sys.path.insert(0, str(REPO_DIR / "kg_interface"))
from litrag import KnowledgeGraph, RetrievalEnv, train_policy

MAX_STEPS = 4
STEP_COST = 0.10
K = 3
LINK_ENTITIES = True
TRAIN_LIMIT = 1000  # Set to None to use all train rows; expected to take much longer.
EPOCHS = 2

kg = KnowledgeGraph(str(KG_DIR))
train_qa = [qa for qa in kg.qa if qa.get("split") == "train"]
dev_qa = [qa for qa in kg.qa if qa.get("split") == "dev"]
assert train_qa and dev_qa, "Converted data must contain train and dev QA rows."

training_rows = train_qa if TRAIN_LIMIT is None else train_qa[:TRAIN_LIMIT]
env = RetrievalEnv(kg, max_steps=MAX_STEPS, step_cost=STEP_COST)
policy = train_policy(
    kg, env, training_rows, epochs=EPOCHS, link=LINK_ENTITIES, verbose=True
)
policy.save(str(POLICY_PATH))
PARAMS = {
    "max_steps": MAX_STEPS,
    "step_cost": STEP_COST,
    "k": K,
    "link": LINK_ENTITIES,
}
PARAMS_PATH.write_text(json.dumps(PARAMS, indent=2), encoding="utf-8")
print(f"Trained on {len(training_rows):,} of {len(train_qa):,} train rows")
print(f"Full validation questions to evaluate: {len(dev_qa):,}")
print("Policy:", POLICY_PATH)


## 7. Evaluate every validation question and save the report

This cell runs dense, never-expand, always-expand, and learned-RL retrieval over
the entire `dev` split. It may take hours; do not set a `limit` here.


In [ ]:
from rag_metrics import evaluate_system, write_reports

results = evaluate_system(kg, dev_qa, K, policy=policy, params=PARAMS)
json_path, csv_path = write_reports(results, METRICS_DIR)
print(json.dumps(results, indent=2))
print("JSON report:", json_path)
print("CSV report:", csv_path)


## 8. Download or inspect outputs

The following files are in Google Drive under `MyDrive/LiteRag_Full_Run/`:

- `data/`: full converted graph, sentence chunks, and QA records
- `policy.npz`: trained policy weights
- `rl_params.json`: policy/environment parameters
- `metrics/rag_metrics.json` and `metrics/rag_metrics.csv`: full-dev metrics

These are retrieval metrics (supporting-fact retrieval and answer-text presence),
not exact-match/F1 or faithfulness metrics from a generated answer. The current
pipeline does not include an answer-generation model.
